# Predicción de Suscripción a Depósitos Bancarios mediante Aprendizaje Automático Supervisado

**Curso:** Manejo de Datos para la IA — Postgrado de Inteligencia Artificial, Universidad Galileo
**Integrante:** Diego Fernando Alvarez Muñoz
**Tipo de problema:** Clasificación binaria supervisada
**Dataset:** Bank Marketing — `bank-additional-full.csv` (UCI Machine Learning Repository)
**Variable objetivo:** `y` (`yes` = el cliente suscribió un depósito a plazo / `no` = no lo suscribió)

---

### Pregunta de investigación
¿Es posible predecir, **antes de contactar al cliente**, si aceptará un depósito a plazo, utilizando características del cliente, historial de campañas y contexto socioeconómico, y con qué utilidad de negocio?

### Hoja de ruta del notebook
0. Configuración y reproducibilidad
1. Carga, inventario y selección del dataset
2. EDA — estructura y balance de clases
3. EDA — diagnóstico de calidad de datos
4. EDA — análisis inferencial (significancia y colinealidad)
5. Preprocesamiento sin fuga de información (Pipeline)
6. Estrategia de validación (incluye control de fuga temporal)
7. Modelado: pool de candidatos, tuning y stacking
8. Evaluación comparativa (PR-AUC, ROC-AUC, KS, Lift, calibración)
9. Comparación estadística entre modelos
10. Ajuste de umbral con lectura de negocio
11. Interpretabilidad (SHAP + permutation importance)
12. Modelo final, guardado de artefactos y conclusiones
13. Apéndice de reproducibilidad

> **Nota metodológica central.** La variable `duration` (duración de la llamada) solo se conoce *después* de contactar al cliente, por lo que **no está disponible al momento de predecir**. Por eso entrenamos y evaluamos todo en **dos escenarios**: *CON* `duration` (benchmark, comparable con la literatura) y *SIN* `duration` (escenario realista de despliegue). Esta decisión sigue la advertencia explícita del repositorio UCI y el trabajo de Moro, Cortez & Rita (2014).

## 0. Configuración y reproducibilidad

In [ ]:
# ============================================================
# 0. Configuración general y semillas
# ============================================================
from pathlib import Path
import zipfile, warnings, sys, platform, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 120)
plt.rcParams['figure.dpi'] = 130

# Paleta y estilo de figuras (consistente con el paper)
NAVY, GOLD, GREEN, GREY, RED = '#1F3864', '#C9A227', '#2E8B57', '#9AA0A6', '#C0392B'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titleweight': 'bold', 'font.size': 11})

# Carpetas de salida para artefactos del paper
FIG_DIR = Path('figuras_paper'); FIG_DIR.mkdir(exist_ok=True)
MODEL_DIR = Path('modelos'); MODEL_DIR.mkdir(exist_ok=True)

def guardar_fig(nombre):
    '''Guarda la figura actual en figuras_paper/ para incorporarla al paper.'''
    plt.savefig(FIG_DIR / f'{nombre}.png', bbox_inches='tight', dpi=160, facecolor='white')

print('Python', sys.version.split()[0], '|', platform.system())
print('Semilla global:', RANDOM_STATE)

### Dependencias
Este notebook está pensado para ejecutarse en una máquina local. Requiere `scikit-learn`, `scipy`, `xgboost` y `shap` además de la pila estándar de datos. Si falta alguna:

```bash
pip install scikit-learn scipy xgboost shap matplotlib pandas numpy
```

In [ ]:
# Verificación de dependencias clave (no instala nada; solo informa)
import importlib
for pkg in ['sklearn', 'scipy', 'xgboost', 'shap']:
    try:
        m = importlib.import_module(pkg)
        print(f'{pkg:10s} {getattr(m, "__version__", "?")}')
    except ImportError:
        print(f'{pkg:10s} NO INSTALADO  ->  pip install {pkg}')

## 1. Carga, inventario y selección del dataset

El conjunto Bank Marketing incluye cuatro versiones. La estrategia de carga es robusta: busca primero los CSV ya extraídos y, si no los encuentra, descomprime cualquier ZIP de Bank Marketing presente en la carpeta (incluyendo ZIPs anidados).

In [ ]:
# ============================================================
# 1.1 Localización y extracción robusta de los datos
# ============================================================
BASE_DIR = Path('bank_marketing_data')
BASE_DIR.mkdir(exist_ok=True)

def localizar_csvs():
    return sorted(set(Path('.').rglob('bank*additional*full.csv')) |
                  set(BASE_DIR.rglob('*.csv')))

# Si aún no hay CSV, intentar extraer cualquier ZIP de bank marketing
if not localizar_csvs():
    zips = list(Path('.').rglob('*bank*marketing*.zip')) + list(Path('.').rglob('bank*additional*.zip'))
    if not zips:
        raise FileNotFoundError(
            'No se encontraron CSV ni ZIP de Bank Marketing. '
            'Coloca el archivo bank_marketing.zip en la carpeta del notebook.')
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall(BASE_DIR)
    # Extraer ZIPs anidados
    for nz in BASE_DIR.rglob('*.zip'):
        with zipfile.ZipFile(nz) as zf:
            zf.extractall(nz.with_suffix(''))

csv_paths = localizar_csvs()
print('CSV encontrados:')
for p in csv_paths:
    print('  -', p, f'({p.stat().st_size/1024:.0f} KB)')

In [ ]:
# ============================================================
# 1.2 Perfil comparativo de las versiones del dataset
# ============================================================
datasets = {}
for path in csv_paths:
    try:
        datasets[path.name] = pd.read_csv(path, sep=';')
    except Exception as e:
        print('No se pudo leer', path, e)

filas = []
for nombre, d in datasets.items():
    unk = sum((d[c].astype(str).str.lower() == 'unknown').sum()
              for c in d.select_dtypes(include='object').columns)
    filas.append({
        'dataset': nombre,
        'filas': d.shape[0], 'columnas': d.shape[1],
        'nulos': int(d.isna().sum().sum()),
        'unknown': int(unk),
        'duplicados': int(d.duplicated().sum()),
        'tiene_y': 'y' in d.columns,
    })
comparativo = pd.DataFrame(filas).sort_values(['columnas', 'filas'], ascending=False).reset_index(drop=True)
comparativo

**Justificación de la selección.** Trabajamos con **`bank-additional-full.csv`** (41,188 filas × 21 columnas). Aunque `bank-full.csv` tiene más registros (45,211), carece de las **variables socioeconómicas** (`euribor3m`, `emp.var.rate`, `nr.employed`, etc.), que en Moro, Cortez & Rita (2014) resultaron ser de las más predictivas (el Euribor a 3 meses fue el atributo #1). La versión completa enriquecida es, por tanto, la más adecuada para el objetivo.

In [ ]:
DATASET_PRINCIPAL = 'bank-additional-full.csv'
assert DATASET_PRINCIPAL in datasets, f'No se encontró {DATASET_PRINCIPAL}'
df_raw = datasets[DATASET_PRINCIPAL].copy()
print(f'Dataset principal: {DATASET_PRINCIPAL}  ->  {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas')
df_raw.head()

## 2. EDA — estructura y balance de clases

In [ ]:
# ============================================================
# 2.1 Tipos de variable y agrupación conceptual
# ============================================================
info_vars = pd.DataFrame({
    'variable': df_raw.columns,
    'tipo': [str(df_raw[c].dtype) for c in df_raw.columns],
    'n_unicos': [df_raw[c].nunique() for c in df_raw.columns],
    'nulos': [int(df_raw[c].isna().sum()) for c in df_raw.columns],
})

grupos = {
    'Cliente': ['age','job','marital','education','default','housing','loan'],
    'Contacto y campaña': ['contact','month','day_of_week','duration','campaign','pdays','previous','poutcome'],
    'Contexto socioeconómico': ['emp.var.rate','cons.price.idx','cons.conf.idx','euribor3m','nr.employed'],
    'Objetivo': ['y'],
}
info_vars['grupo'] = info_vars['variable'].map({v:g for g,vs in grupos.items() for v in vs})
info_vars

In [ ]:
# ============================================================
# 2.2 Balance de la variable objetivo
# ============================================================
balance = (df_raw['y'].value_counts().rename_axis('clase').reset_index(name='cantidad'))
balance['porcentaje'] = (balance['cantidad'] / len(df_raw) * 100).round(2)
print(balance.to_string(index=False))

fig, ax = plt.subplots(figsize=(5.2, 3.6))
bars = ax.bar(balance['clase'].astype(str), balance['cantidad'], color=[NAVY, GOLD])
ax.set_title('Balance de la variable objetivo'); ax.set_ylabel('Registros')
for b, pct in zip(bars, balance['porcentaje']):
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), f'{pct}%', ha='center', va='bottom', fontweight='bold')
guardar_fig('fig1_balance'); plt.show()

El dataset está **fuertemente desbalanceado**: ~88.7% `no` frente a ~11.3% `yes`. Implicación directa: *accuracy* es engañosa (un clasificador trivial que prediga siempre `no` alcanza ~89%). La evaluación se centrará en **PR-AUC**, recall de la clase positiva y métricas de negocio (Lift).

## 3. EDA — diagnóstico de calidad de datos

In [ ]:
# ============================================================
# 3.1 Valores 'unknown' y nulos
# ============================================================
rep = []
for col in df_raw.columns:
    unk = int((df_raw[col].astype(str).str.lower() == 'unknown').sum()) if df_raw[col].dtype == 'object' else 0
    rep.append({'variable': col, 'nulos': int(df_raw[col].isna().sum()),
                'unknown': unk, 'pct_unknown': round(unk/len(df_raw)*100, 2)})
unknown_report = pd.DataFrame(rep).sort_values('pct_unknown', ascending=False).reset_index(drop=True)
print('No hay nulos formales (NaN):', df_raw.isna().sum().sum() == 0)

plot = unknown_report[unknown_report['unknown'] > 0].sort_values('unknown')
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.barh(plot['variable'], plot['unknown'], color=NAVY)
for i, (v, p) in enumerate(zip(plot['unknown'], plot['pct_unknown'])):
    ax.text(v, i, f' {p}%', va='center')
ax.set_title("Valores 'unknown' por variable"); ax.set_xlabel('Cantidad')
guardar_fig('fig2_unknown'); plt.show()
unknown_report

In [ ]:
# ============================================================
# 3.2 La variable 'default' es casi constante: hallazgo clave
# ============================================================
print('Distribución de default:')
print(df_raw['default'].value_counts().to_string())
print(f"\n-> Solo {(df_raw['default']=='yes').sum()} registros 'yes' de {len(df_raw):,}.")
print("-> Más allá del 20.9% de 'unknown', la variable carece de poder discriminante")
print("   (es prácticamente no/unknown). Se DESCARTARÁ en el preprocesamiento.")

In [ ]:
# ============================================================
# 3.3 Duplicados
# ============================================================
dups = int(df_raw.duplicated().sum())
print(f'Registros duplicados exactos: {dups}  (se eliminarán en el preprocesamiento)')

In [ ]:
# ============================================================
# 3.4 pdays = 999 (no contactado) y redundancia con previous/poutcome
# ============================================================
print(f"pdays == 999: {(df_raw['pdays']==999).sum():,} ({(df_raw['pdays']==999).mean()*100:.2f}%)")
print('999 NO es un valor real: codifica "cliente no contactado previamente".')
print('\nConsistencia pdays(=999) vs poutcome:')
print(pd.crosstab(df_raw['pdays']==999, df_raw['poutcome']))
print('\nprevious>0 vs pdays==999 (redundancia/solapamiento de información):')
print(pd.crosstab(df_raw['previous']==0, df_raw['pdays']==999,
                  rownames=['previous==0'], colnames=['pdays==999']))
print('\n-> Decisión: derivar variable binaria contacted_prev = (pdays != 999).')
print('   pdays/previous/poutcome codifican parcialmente lo mismo; mantenemos')
print('   previous y poutcome, sustituimos pdays por la binaria para evitar el 999 espurio.')

In [ ]:
# ============================================================
# 3.5 'duration': señal fuerte pero con fuga de información
# ============================================================
print('Duración media por clase (segundos):')
print(df_raw.groupby('y')['duration'].mean().round(1).to_string())
print(f"\nRegistros con duration == 0: {(df_raw['duration']==0).sum()} "
      f"-> y de esos: {df_raw.loc[df_raw['duration']==0,'y'].unique().tolist()}")
print('\n-> Evidencia de fuga: duration=0 implica y=no SIEMPRE (no hubo llamada).')
print('   duration solo se conoce TRAS la llamada -> no usable para predecir.')
print('   Por eso evaluaremos CON y SIN duration (benchmark vs realista).')

## 4. EDA — análisis inferencial

El EDA puramente descriptivo no distingue una diferencia real de una casualidad muestral. Aquí añadimos pruebas de significancia y medimos la colinealidad, que es crítica para la interpretabilidad de la Regresión Logística.

In [ ]:
# ============================================================
# 4.1 Categóricas vs y: Chi-cuadrado + V de Cramér
# ============================================================
from scipy.stats import chi2_contingency

def cramers_v(conf):
    chi2 = chi2_contingency(conf)[0]
    n = conf.sum().sum()
    r, k = conf.shape
    return np.sqrt((chi2/n) / (min(r-1, k-1)))

cat_cols = ['job','marital','education','housing','loan','contact','month','day_of_week','poutcome']
res = []
for c in cat_cols:
    conf = pd.crosstab(df_raw[c], df_raw['y'])
    chi2, p, dof, _ = chi2_contingency(conf)
    res.append({'variable': c, 'chi2': round(chi2,1), 'p_value': p,
                'cramers_v': round(cramers_v(conf),3),
                'significativa_(p<0.05)': p < 0.05})
chi_tabla = pd.DataFrame(res).sort_values('cramers_v', ascending=False).reset_index(drop=True)
chi_tabla

In [ ]:
# ============================================================
# 4.2 Numéricas vs y: Mann-Whitney U + Kolmogorov-Smirnov (2 muestras)
# ============================================================
from scipy.stats import mannwhitneyu, ks_2samp

num_cols = ['age','duration','campaign','pdays','previous',
            'emp.var.rate','cons.price.idx','cons.conf.idx','euribor3m','nr.employed']
g_yes = df_raw[df_raw['y']=='yes']; g_no = df_raw[df_raw['y']=='no']
res = []
for c in num_cols:
    u, p_mw = mannwhitneyu(g_yes[c], g_no[c], alternative='two-sided')
    ks, p_ks = ks_2samp(g_yes[c], g_no[c])
    res.append({'variable': c,
                'media_yes': round(g_yes[c].mean(),2), 'media_no': round(g_no[c].mean(),2),
                'mannwhitney_p': p_mw, 'ks_stat': round(ks,3), 'ks_p': p_ks,
                'sig_(p<0.05)': (p_mw < 0.05)})
num_tabla = pd.DataFrame(res).sort_values('ks_stat', ascending=False).reset_index(drop=True)
num_tabla

In [ ]:
# ============================================================
# 4.3 Colinealidad de variables económicas + VIF
# ============================================================
econ = ['emp.var.rate','cons.price.idx','cons.conf.idx','euribor3m','nr.employed']
corr = df_raw[econ].corr()

fig, ax = plt.subplots(figsize=(5.6, 4.6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(econ))); ax.set_xticklabels(econ, rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(len(econ))); ax.set_yticklabels(econ, fontsize=9)
for i in range(len(econ)):
    for j in range(len(econ)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if abs(corr.iloc[i,j])>0.5 else 'black')
ax.set_title('Correlación entre variables económicas')
plt.colorbar(im, fraction=0.046, pad=0.04)
guardar_fig('fig3_colinealidad'); plt.show()

# VIF = diagonal de la inversa de la matriz de correlación
vif = pd.Series(np.diag(np.linalg.inv(corr.values)), index=econ).round(2)
print('VIF (regla práctica: >5 colinealidad alta, >10 severa):')
print(vif.to_string())
print('\n-> euribor3m, emp.var.rate y nr.employed están casi perfectamente correlacionadas.')
print('   Impacto: coeficientes inestables en LR. Mitigación: regularización L2 (la LR la usa)')
print('   y modelos de árbol robustos a colinealidad. Lo discutimos en el paper.')

### 4.4 Tasas de conversión por variables categóricas relevantes

In [ ]:
# Figura EDA: conversión por resultado de campaña previa y por mes (estilo paper)
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
po = df_raw.groupby('poutcome')['y'].apply(lambda s: (s == 'yes').mean()*100).sort_values()
axs[0].barh(po.index, po.values, color=GOLD)
for i, v in enumerate(po.values): axs[0].text(v, i, f' {v:.1f}%', va='center')
axs[0].set_title('Tasa de suscripción por resultado previo')
orden = ['mar','sep','oct','dec','apr','aug','jun','nov','jul','may']
orden = [m for m in orden if m in df_raw['month'].unique()]
mo = df_raw.groupby('month')['y'].apply(lambda s: (s == 'yes').mean()*100).reindex(orden)
axs[1].bar(mo.index, mo.values, color=NAVY); axs[1].set_ylabel('%')
axs[1].set_title('Tasa de suscripción por mes')
guardar_fig('fig4_eda'); plt.show()

## 5. Preprocesamiento sin fuga de información

**Principio rector:** *toda* transformación que aprende de los datos (escalado, codificación, eventual remuestreo) vive **dentro de un `Pipeline`**, de modo que se ajusta solo con el *fold* de entrenamiento en cada partición del CV. Esto evita la fuga que aparece, por ejemplo, cuando se remuestrea o escala sobre el dataset completo antes de partir (un error frecuente en la literatura sobre este dataset).

Decisiones de limpieza:
- Eliminar 12 duplicados exactos.
- Descartar `default` (casi constante).
- Derivar `contacted_prev = (pdays != 999)` y descartar `pdays`.
- `unknown` se conserva como **categoría propia** (es informativa; no se imputa).
- One-Hot para categóricas; escalado **solo** para modelos sensibles a escala (LR, LinearSVC).

In [ ]:
# ============================================================
# 5.1 Limpieza base y construcción de los dos escenarios de features
# ============================================================
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

def preparar_base(df_in):
    d = df_in.drop_duplicates().reset_index(drop=True)
    d['contacted_prev'] = (d['pdays'] != 999).astype(int)
    d = d.drop(columns=['default', 'pdays'])
    return d

df = preparar_base(df_raw)
y = (df['y'] == 'yes').astype(int)
print('Tras limpieza:', df.shape, '| positivos:', int(y.sum()), f'({y.mean()*100:.2f}%)')

NUM_BASE = ['age','campaign','previous','emp.var.rate','cons.price.idx',
            'cons.conf.idx','euribor3m','nr.employed','contacted_prev']
CAT_BASE = ['job','marital','education','housing','loan','contact','month','day_of_week','poutcome']

def features(con_duration: bool):
    num = NUM_BASE + (['duration'] if con_duration else [])
    return num, CAT_BASE

def preprocesador(num, cat, escalar: bool):
    '''escalar=True para LR/LinearSVC; False para árboles.'''
    num_t = StandardScaler() if escalar else 'passthrough'
    return ColumnTransformer([
        ('num', num_t, num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat),
    ])

print('Escenario realista (sin duration):', len(features(False)[0]), 'num +', len(features(False)[1]), 'cat')
print('Escenario benchmark (con duration):', len(features(True)[0]), 'num +', len(features(True)[1]), 'cat')

## 6. Estrategia de validación

1. **Partición principal:** *train/test* estratificado 75/25, y `StratifiedKFold(5)` para CV y tuning sobre el *train*. Es el protocolo estándar esperado.
2. **Control de fuga temporal:** las filas vienen ordenadas cronológicamente (mayo 2008 en adelante) y las variables económicas son casi constantes dentro de cada periodo, por lo que un split aleatorio puede "ver el futuro". Añadimos un **holdout ordenado por posición** (primeras filas → entrenar, últimas → probar) y comparamos. Si las métricas caen, confirmamos el efecto temporal y lo reportamos como limitación (en línea con el esquema *rolling windows* de Moro 2014).
3. **Desbalance:** `class_weight='balanced'` (LR, LinearSVC, RF) y `scale_pos_weight` (XGB), más **ajuste de umbral**. Preferimos esto a remuestreo agresivo; si se usara SMOTE, iría dentro del pipeline.

In [ ]:
# ============================================================
# 6.1 Particiones
# ============================================================
from sklearn.model_selection import train_test_split, StratifiedKFold

# Datos completos por escenario
def Xy(con_duration):
    num, cat = features(con_duration)
    return df[num + cat], y

# Partición estratificada (escenario realista por defecto)
X_real, _ = Xy(False)
X_bench, _ = Xy(True)
Xtr_r, Xte_r, ytr, yte = train_test_split(X_real, y, test_size=0.25,
                                           stratify=y, random_state=RANDOM_STATE)
Xtr_b, Xte_b = X_bench.loc[Xtr_r.index], X_bench.loc[Xte_r.index]

# Holdout ordenado por posición (aprox. temporal) — escenario realista
corte = int(len(X_real) * 0.75)
Xtr_t, Xte_t = X_real.iloc[:corte], X_real.iloc[corte:]
ytr_t, yte_t = y.iloc[:corte], y.iloc[corte:]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
spw = (ytr == 0).sum() / (ytr == 1).sum()   # scale_pos_weight para XGB
print(f'Train: {len(Xtr_r):,} | Test: {len(Xte_r):,} | scale_pos_weight={spw:.2f}')

## 7. Modelado: pool de candidatos, tuning y stacking

Pool: **Regresión Logística** (línea base interpretable), **LinearSVC** (comparación rápida tipo SVM lineal, calibrado para producir probabilidades), **Random Forest** y **XGBoost**. Se afinan RF y XGB con `RandomizedSearchCV` (optimizando **PR-AUC**) dentro del CV del *train*, y se combinan en un **StackingClassifier** con meta-modelo logístico.

In [ ]:
# ============================================================
# 7.1 Definición de modelos (factoría por escenario)
# ============================================================
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier

def construir_modelos(con_duration):
    num, cat = features(con_duration)
    pre_s  = lambda: preprocesador(num, cat, escalar=True)
    pre_ns = lambda: preprocesador(num, cat, escalar=False)
    return {
        'LR': Pipeline([('pre', pre_s()),
                        ('clf', LogisticRegression(max_iter=2000, class_weight='balanced',
                                                   random_state=RANDOM_STATE))]),
        'LinearSVC': Pipeline([('pre', pre_s()),
                        ('clf', CalibratedClassifierCV(
                            LinearSVC(class_weight='balanced', dual='auto',
                                      random_state=RANDOM_STATE), cv=3))]),
        'RandomForest': Pipeline([('pre', pre_ns()),
                        ('clf', RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                       n_jobs=-1, random_state=RANDOM_STATE))]),
        'XGBoost': Pipeline([('pre', pre_ns()),
                        ('clf', XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.1,
                                              subsample=0.9, colsample_bytree=0.9,
                                              scale_pos_weight=spw, eval_metric='logloss',
                                              random_state=RANDOM_STATE, n_jobs=-1))]),
    }

modelos_real = construir_modelos(False)
print('Modelos definidos (escenario realista):', list(modelos_real))

In [ ]:
# ============================================================
# 7.2 Tuning de RF y XGB con RandomizedSearchCV (optimiza PR-AUC)
# ============================================================
from sklearn.model_selection import RandomizedSearchCV

espacios = {
    'RandomForest': {
        'clf__n_estimators': [200, 400, 600],
        'clf__max_depth': [None, 8, 12, 20],
        'clf__min_samples_leaf': [1, 5, 20],
        'clf__max_features': ['sqrt', 'log2', 0.5],
    },
    'XGBoost': {
        'clf__n_estimators': [200, 400, 600],
        'clf__max_depth': [3, 4, 6, 8],
        'clf__learning_rate': [0.03, 0.05, 0.1],
        'clf__subsample': [0.7, 0.9, 1.0],
        'clf__colsample_bytree': [0.7, 0.9, 1.0],
        'clf__min_child_weight': [1, 3, 5],
    },
}

def tunear(modelos, n_iter=20):
    afinados = dict(modelos)
    for nombre, espacio in espacios.items():
        print(f'Tuning {nombre} ...', end=' ')
        rs = RandomizedSearchCV(modelos[nombre], espacio, n_iter=n_iter,
                                scoring='average_precision', cv=cv,
                                random_state=RANDOM_STATE, n_jobs=-1, refit=True)
        rs.fit(Xtr_r, ytr)
        afinados[nombre] = rs.best_estimator_
        print(f'PR-AUC(cv)={rs.best_score_:.3f}')
    return afinados

# Sugerencia: n_iter=20 (sube a 40-60 si tu equipo lo permite)
modelos_real = tunear(modelos_real, n_iter=20)

In [ ]:
# ============================================================
# 7.3 Ensemble por stacking (meta-modelo logístico)
# ============================================================
stack_real = StackingClassifier(
    estimators=[('lr', modelos_real['LR']),
                ('rf', modelos_real['RandomForest']),
                ('xgb', modelos_real['XGBoost'])],
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    stack_method='predict_proba', cv=5, n_jobs=-1)

modelos_real['Stacking'] = stack_real
print('Ajustando todos los modelos del escenario realista...')
for nombre, m in modelos_real.items():
    m.fit(Xtr_r, ytr)
    print(' -', nombre, 'OK')

## 8. Evaluación comparativa

Métricas: **PR-AUC** (principal, por el desbalance), **ROC-AUC**, **KS** (separación de scores, estándar en *scoring* bancario), precision/recall/F1 y accuracy (referencia). Más matriz de confusión, **curva Lift/ganancias acumuladas** (la métrica de negocio de esta literatura) y **calibración**.

In [ ]:
# ============================================================
# 8.1 Funciones de evaluación
# ============================================================
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score, confusion_matrix,
    brier_score_loss)
from scipy.stats import ks_2samp

def ks_score(y_true, p):
    return ks_2samp(p[y_true == 1], p[y_true == 0]).statistic

def evaluar(modelos, Xte, yte, umbral=0.5):
    filas = []
    for nombre, m in modelos.items():
        p = m.predict_proba(Xte)[:, 1]
        pred = (p >= umbral).astype(int)
        filas.append({
            'modelo': nombre,
            'PR_AUC': round(average_precision_score(yte, p), 4),
            'ROC_AUC': round(roc_auc_score(yte, p), 4),
            'KS': round(ks_score(yte.values, p), 4),
            'F1': round(f1_score(yte, pred), 4),
            'Precision': round(precision_score(yte, pred), 4),
            'Recall': round(recall_score(yte, pred), 4),
            'Accuracy': round(accuracy_score(yte, pred), 4),
            'Brier': round(brier_score_loss(yte, p), 4),
        })
    return pd.DataFrame(filas).sort_values('PR_AUC', ascending=False).reset_index(drop=True)

tabla_real = evaluar(modelos_real, Xte_r, yte)
print('=== Escenario REALISTA (sin duration) ==='); print(tabla_real.to_string(index=False))

In [ ]:
# ============================================================
# 8.2 Escenario benchmark CON duration (entrena y evalúa el pool)
# ============================================================
modelos_bench = construir_modelos(True)
modelos_bench = {k: v for k, v in modelos_bench.items()}  # sin tuning extra para agilizar
modelos_bench['Stacking'] = StackingClassifier(
    estimators=[('lr', modelos_bench['LR']),
                ('rf', modelos_bench['RandomForest']),
                ('xgb', modelos_bench['XGBoost'])],
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    stack_method='predict_proba', cv=5, n_jobs=-1)
for m in modelos_bench.values():
    m.fit(Xtr_b, ytr)

tabla_bench = evaluar(modelos_bench, Xte_b, yte)
print('=== Escenario BENCHMARK (con duration) ==='); print(tabla_bench.to_string(index=False))
print('\n-> La caída de PR-AUC/KS al quitar duration cuantifica cuánta "señal" venía')
print('   de una variable no disponible en producción. El modelo válido es el realista.')

In [ ]:
# ============================================================
# 8.3 Control de fuga temporal: holdout ordenado vs estratificado
# ============================================================
xgb_temporal = construir_modelos(False)['XGBoost']
xgb_temporal.fit(Xtr_t, ytr_t)
p_temp = xgb_temporal.predict_proba(Xte_t)[:, 1]
print('XGBoost — comparación de protocolos (escenario realista):')
print(f"  Split estratificado : PR-AUC={tabla_real.loc[tabla_real['modelo']=='XGBoost','PR_AUC'].values[0]:.3f}")
print(f"  Holdout temporal    : PR-AUC={average_precision_score(yte_t, p_temp):.3f} | "
      f"ROC={roc_auc_score(yte_t, p_temp):.3f} | KS={ks_score(yte_t.values, p_temp):.3f}")
print('\n-> Una diferencia notable confirma sensibilidad al ordenamiento temporal')
print('   (variables económicas como proxy del tiempo). Se reporta como limitación.')

In [ ]:
# ============================================================
# 8.4 Mejor modelo: matriz de confusión (estilo paper)
# ============================================================
from sklearn.calibration import calibration_curve

mejor_nombre = tabla_real.iloc[0]['modelo']
mejor = modelos_real[mejor_nombre]
p_best = mejor.predict_proba(Xte_r)[:, 1]
print('Mejor modelo (PR-AUC):', mejor_nombre)

cm = confusion_matrix(yte, (p_best >= 0.5).astype(int))
fig, ax = plt.subplots(figsize=(4, 3.6)); im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center', fontsize=13,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_xticks([0,1]); ax.set_xticklabels(['no','yes']); ax.set_xlabel('Predicho')
ax.set_yticks([0,1]); ax.set_yticklabels(['no','yes']); ax.set_ylabel('Real')
ax.set_title(f'Matriz de confusión - {mejor_nombre}')
guardar_fig('fig6_confusion'); plt.show()

In [ ]:
# Curva de ganancias acumuladas (Lift) - estilo paper
colores = {'Stacking': NAVY, 'RandomForest': GOLD, 'XGBoost': GREEN, 'LR': GREY}
nombres = {'Stacking':'Stacking','RandomForest':'Random Forest','XGBoost':'XGBoost','LR':'Reg. Logística'}
fig, ax = plt.subplots(figsize=(6, 4.6))
for n in ['Stacking', 'RandomForest', 'XGBoost', 'LR']:
    if n in modelos_real:
        p = modelos_real[n].predict_proba(Xte_r)[:, 1]
        d = pd.DataFrame({'y': yte.values, 'p': p}).sort_values('p', ascending=False)
        ax.plot(np.arange(1, len(d)+1)/len(d), d['y'].cumsum()/d['y'].sum(),
                label=nombres[n], color=colores[n], lw=2)
ax.plot([0,1], [0,1], '--', color='#888', label='aleatorio')
ax.axvline(0.5, color=RED, ls=':', lw=1.2)
ax.set_xlabel('Fracción de clientes contactados (ordenados por score)')
ax.set_ylabel('Suscriptores capturados')
ax.set_title('Curva de ganancias acumuladas (Lift)'); ax.legend(fontsize=9)
guardar_fig('fig5_lift'); plt.show()

# Lectura de negocio
d = pd.DataFrame({'y': yte.values, 'p': p_best}).sort_values('p', ascending=False)
for frac in [0.2, 0.5]:
    n = int(len(d)*frac); capt = d.head(n)['y'].sum() / d['y'].sum()
    print(f'Contactando al {int(frac*100)}% mejor rankeado se captura el {capt*100:.1f}% de los suscriptores.')

In [ ]:
# Calibración del mejor modelo (estilo paper)
frac_pos, mean_pred = calibration_curve(yte, p_best, n_bins=10, strategy='quantile')
fig, ax = plt.subplots(figsize=(4.8, 4.4))
ax.plot(mean_pred, frac_pos, 'o-', color=NAVY, label=mejor_nombre)
ax.plot([0,1], [0,1], '--', color='#888', label='ideal')
ax.set_xlabel('Probabilidad predicha'); ax.set_ylabel('Frecuencia observada')
ax.set_title(f'Calibración (Brier={brier_score_loss(yte,p_best):.3f})'); ax.legend(fontsize=9)
guardar_fig('fig7_calibracion'); plt.show()

## 9. Comparación estadística entre modelos

"Mayor AUC" no basta: la diferencia puede deberse al azar de la partición. Estimamos PR-AUC por *fold* con **CV repetido** sobre los modelos individuales y aplicamos la prueba pareada de **Mann-Whitney** (mismo enfoque que Moro 2014).

In [ ]:
# ============================================================
# 9.1 PR-AUC por fold (CV repetido) + prueba pareada Mann-Whitney
# ============================================================
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from scipy.stats import mannwhitneyu

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
scores = {}
for nombre in ['LR', 'LinearSVC', 'RandomForest', 'XGBoost']:
    s = cross_val_score(modelos_real[nombre], Xtr_r, ytr,
                        scoring='average_precision', cv=rcv, n_jobs=-1)
    scores[nombre] = s
    print(f'{nombre:13s} PR-AUC = {s.mean():.4f} ± {s.std():.4f}')

mejor_indiv = max(scores, key=lambda k: scores[k].mean())
print(f'\nPrueba pareada (Mann-Whitney) de {mejor_indiv} vs el resto:')
for nombre, s in scores.items():
    if nombre == mejor_indiv: continue
    _, p = mannwhitneyu(scores[mejor_indiv], s, alternative='greater')
    print(f'  {mejor_indiv} > {nombre}: p={p:.4f} '
          f'{"(significativo)" if p < 0.05 else "(no significativo)"}')

## 10. Ajuste de umbral con lectura de negocio

El umbral 0.5 rara vez es óptimo con clases desbalanceadas. Elegimos el umbral que maximiza F1 sobre el *train* (vía CV) y mostramos el equilibrio precision/recall, que el negocio puede mover según su capacidad de contacto.

In [ ]:
# ============================================================
# 10.1 Selección de umbral (maximiza F1) sin tocar el test
# ============================================================
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve

p_cv = cross_val_predict(mejor, Xtr_r, ytr, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
prec, rec, thr = precision_recall_curve(ytr, p_cv)
f1s = 2*prec*rec / (prec+rec+1e-12)
umbral_opt = thr[np.argmax(f1s[:-1])]
print(f'Umbral óptimo (máx F1 en CV): {umbral_opt:.3f}')

pred_opt = (p_best >= umbral_opt).astype(int)
print(f'En test con umbral {umbral_opt:.3f}: '
      f'Precision={precision_score(yte,pred_opt):.3f} '
      f'Recall={recall_score(yte,pred_opt):.3f} '
      f'F1={f1_score(yte,pred_opt):.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(thr, prec[:-1], label='Precision', color=NAVY)
ax.plot(thr, rec[:-1], label='Recall', color=GOLD)
ax.plot(thr, f1s[:-1], label='F1', color=RED)
ax.axvline(umbral_opt, color='#444', ls='--', label=f'umbral óptimo={umbral_opt:.2f}')
ax.set_xlabel('Umbral de decisión'); ax.set_ylabel('Métrica'); ax.legend(fontsize=9)
ax.set_title('Precision / Recall / F1 vs umbral')
guardar_fig('fig9_umbral'); plt.show()

## 11. Interpretabilidad

Abrimos la "caja negra" con tres lentes complementarias: **SHAP (TreeSHAP)** sobre XGBoost — el equivalente moderno al análisis de sensibilidad y curvas VEC de Moro 2014 —, **permutation importance** (model-agnóstica) y los **coeficientes de la Regresión Logística**.

In [ ]:
# ============================================================
# 11.1 SHAP (TreeSHAP) sobre XGBoost (estilo paper)
# ============================================================
import shap

xgb_pipe = modelos_real['XGBoost']
pre = xgb_pipe.named_steps['pre']
clf = xgb_pipe.named_steps['clf']
feat_names = [f.replace('num__','').replace('cat__','') for f in pre.get_feature_names_out()]

X_shap = Xte_r.sample(min(2000, len(Xte_r)), random_state=RANDOM_STATE)
X_shap_t = pre.transform(X_shap)
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_shap_t)

shap.summary_plot(shap_values, X_shap_t, feature_names=feat_names, show=False, max_display=12, color_bar=True)
fig = plt.gcf(); fig.set_size_inches(7, 4.8)
plt.title('Importancia SHAP - XGBoost', fontweight='bold')
guardar_fig('fig8_shap'); plt.show()

In [ ]:
# Importancia media |SHAP| (barras) — para tabla del paper
imp = pd.DataFrame({'feature': feat_names,
                    'shap_abs_mean': np.abs(shap_values).mean(axis=0)}) \
        .sort_values('shap_abs_mean', ascending=False).head(15).reset_index(drop=True)
print(imp.to_string(index=False))

In [ ]:
# ============================================================
# 11.2 Permutation importance (model-agnóstica) sobre el mejor modelo
# ============================================================
from sklearn.inspection import permutation_importance

perm = permutation_importance(mejor, Xte_r, yte, scoring='average_precision',
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_tabla = pd.DataFrame({'variable': Xte_r.columns,
                           'importancia': perm.importances_mean,
                           'std': perm.importances_std}) \
               .sort_values('importancia', ascending=False).reset_index(drop=True)
perm_tabla.head(15)

In [ ]:
# ============================================================
# 11.3 Coeficientes de la Regresión Logística (signo e interpretación)
# ============================================================
lr_pipe = modelos_real['LR']
lr_feats = lr_pipe.named_steps['pre'].get_feature_names_out()
coefs = pd.DataFrame({'feature': lr_feats,
                      'coef': lr_pipe.named_steps['clf'].coef_[0]})
coefs['abs'] = coefs['coef'].abs()
print('Top +10 (empujan a suscribir):')
print(coefs.sort_values('coef', ascending=False).head(10)[['feature','coef']].to_string(index=False))
print('\nTop -10 (empujan a NO suscribir):')
print(coefs.sort_values('coef').head(10)[['feature','coef']].to_string(index=False))

## 12. Modelo final, artefactos y conclusiones

In [ ]:
# ============================================================
# 12.1 Persistencia del modelo final y de las tablas
# ============================================================
import joblib, json

joblib.dump(mejor, MODEL_DIR / 'modelo_final.joblib')
meta = {'mejor_modelo': mejor_nombre, 'umbral_operativo': float(umbral_opt),
        'escenario': 'realista (sin duration)', 'random_state': RANDOM_STATE,
        'metricas_test': tabla_real.iloc[0].to_dict()}
(MODEL_DIR / 'metadata_modelo.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False))

tabla_real.to_csv(FIG_DIR / 'tabla_resultados_realista.csv', index=False)
tabla_bench.to_csv(FIG_DIR / 'tabla_resultados_benchmark.csv', index=False)
print('Modelo y tablas guardados en', MODEL_DIR, 'y', FIG_DIR)
print(json.dumps(meta, indent=2, ensure_ascii=False))

### Conclusiones (se completan tras ejecutar)

- **Mejor modelo y desempeño realista:** reportar el ganador por PR-AUC en el escenario sin `duration`, con sus métricas (PR-AUC, ROC-AUC, KS) y la comparación estadística de la Sección 9.
- **Valor de negocio:** traducir la curva Lift (p. ej., "contactando al 50% mejor rankeado se captura ~X% de suscriptores"), análogo al hallazgo de Moro 2014.
- **Hallazgos de interpretabilidad:** contrastar el ranking SHAP/permutation con Moro 2014 (se espera `euribor3m`, contacto previo/`poutcome`, variables económicas y mes entre los más relevantes).
- **Limitaciones honestas:**
  1. `duration` no es usable en producción (fuga); el modelo válido es el realista.
  2. Sensibilidad temporal: la brecha entre split estratificado y holdout ordenado (Sección 8.3).
  3. Colinealidad económica (Sección 4.3): limita la interpretación causal de coeficientes.
  4. `default` descartada por falta de poder discriminante.

### Referencias
- Moro, S., Cortez, P., & Rita, P. (2014). *A data-driven approach to predict the success of bank telemarketing.* Decision Support Systems, 62, 22–31.
- Moro, S., Laureano, R., & Cortez, P. (2012). *Enhancing bank direct marketing through data mining.* Proc. 41st EMAC Conference. *(antecedente; usa atributos no disponibles en predicción)*
- Tanvir, M. F., Hossain, M. M., & Jishan, M. A. (2024). *Bayesian Regression for Predicting Subscription to Bank Term Deposits.* arXiv:2410.21539.
- Moro, S., Rita, P., & Cortez, P. (2012). *Bank Marketing.* UCI Machine Learning Repository.

## 13. Apéndice de reproducibilidad

In [ ]:
# Entorno de ejecución (para incluir en el repositorio / requirements.txt)
import sklearn, scipy, xgboost, shap, matplotlib
print('Python      ', sys.version.split()[0])
print('numpy       ', np.__version__)
print('pandas      ', pd.__version__)
print('scikit-learn', sklearn.__version__)
print('scipy       ', scipy.__version__)
print('xgboost     ', xgboost.__version__)
print('shap        ', shap.__version__)
print('matplotlib  ', matplotlib.__version__)
print('\nSemilla global:', RANDOM_STATE)